In [1]:
!pip install nbformat

In [5]:
import nbformat
import os

notebook_name = "your_notebook.ipynb"

# Create a dummy notebook if it doesn't exist for demonstration purposes
if not os.path.exists(notebook_name):
    print(f"Creating a dummy notebook: {notebook_name}")
    dummy_nb = nbformat.v4.new_notebook()
    nbformat.write(dummy_nb, notebook_name)

nb = nbformat.read(notebook_name, as_version=4)

if "widgets" in nb["metadata"]:
    del nb["metadata"]["widgets"]

nbformat.write(nb, "clean_notebook.ipynb")
print(f"Notebook '{notebook_name}' processed and saved to 'clean_notebook.ipynb'")

Creating a dummy notebook: your_notebook.ipynb
Notebook 'your_notebook.ipynb' processed and saved to 'clean_notebook.ipynb'


In [10]:

# SARCASM DETECTION - Full Pipeline


# Install & Imports
!pip install transformers torch scikit-learn pandas -q

import re


import pandas as pd
import requests # Import requests for downloading the file
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from transformers import pipeline

# Dataset Download (if not already present)
dataset_filename = "Sarcasm_Headlines_Dataset.json"
if not pd.io.common.file_exists(dataset_filename):
    print(f"Downloading {dataset_filename}...")
    url = "https://raw.githubusercontent.com/rishabhnmishra/sarcasm-detection-ml/main/Sarcasm_Headlines_Dataset.json"
    response = requests.get(url)
    with open(dataset_filename, 'wb') as f:
        f.write(response.content)
    print("Download complete.")

# Load & Explore Dataset
df = pd.read_json(dataset_filename, lines=True)
print(df.shape)
print(df['is_sarcastic'].value_counts())
df = df[['headline', 'is_sarcastic']]

# Text Cleaning
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z ]', '', text)
    return text

df['clean'] = df['headline'].apply(clean_text)

# TF-IDF + Logistic Regression (ML Model)
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))  # bigrams help catch phrases
X = vectorizer.fit_transform(df['clean'])
y = df['is_sarcastic']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LogisticRegression(max_iter=1000, C=1.0)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print("ML Model Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

# Load Sentiment Model
sentiment_model = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    truncation=True,
    max_length=512
)

# Rule-Based Sarcasm Detection
def is_rule_based_sarcasm(text):
    """
    Detects sarcasm via keyword rules.
    Covers: positive+negative collisions, common sarcastic phrases.
    """
    t = text.lower()

    positive_words = [
        "great", "amazing", "wow", "fantastic", "nice", "perfect",
        "wonderful", "excellent", "brilliant", "superb", "lovely"
    ]
    negative_events = [
        "died", "delay", "bug", "problem", "crashed", "stopped",
        "worst", "broken", "failed", "error", "slow", "terrible",
        "awful", "horrible", "disaster", "again"
    ]

    # Rule 1: positive word + negative event in same sentence
    rule1 = any(p in t for p in positive_words) and any(n in t for n in negative_events)

    # Rule 2: classic sarcastic fixed phrases
    sarcasm_phrases = [
        "just what i needed",
        "perfect timing",
        "love when",
        "love how",
        "oh great",
        "oh sure",
        "oh wow",
        "oh fantastic",
        "oh wonderful",
        "oh perfect",
        "yeah right",
        "as if",
        "totally meant to",
        "so helpful",
        "so useful",
        "what a surprise",
        "what a shock",
        "shocking absolutely shocking",
        "color me surprised",
        "who could have seen that coming",
        "what could go wrong",
        "nothing could go wrong"
    ]
    rule2 = any(p in t for p in sarcasm_phrases)

    return rule1 or rule2


def detect_rhetorical_sarcasm(text):
    """
    Detects rhetorical / conversational sarcasm that keyword rules miss.
    Covers:
      - Rhetorical contradiction: "I'd agree with you but then we'd both be wrong"
      - Mock praise / feigned compliments: "You must know everything"
      - Understatement after negative setup
      - Implied impossibility phrases
    """
    t = text.lower()

    # Pattern 1: Adversative contradiction
    # "I'd agree with you, but then we'd both be wrong"
    rhetorical_contradictions = [
        "i'd agree with you but",
        "i agree but then",
        "i would agree but",
        "sure, if by",
        "yeah, because that worked so well",
        "oh because that makes sense",
        "that makes perfect sense",
        "totally makes sense",
    ]
    if any(p in t for p in rhetorical_contradictions):
        return True

    # Pattern 2: Mock praise / feigned compliments
    # "You must know everything", "Oh you're so smart"
    # Trigger: praise + exaggerated certainty word aimed at a person
    certainty_words = ["must", "surely", "obviously", "clearly", "of course", "definitely"]
    praise_words = ["know everything", "so smart", "so wise", "genius", "expert", "master",
                    "so talented", "so clever", "know it all", "know best"]
    if any(c in t for c in certainty_words) and any(p in t for p in praise_words):
        return True

    # Pattern 3: "Oh, you X? You must Y" structure
    # "Oh you graduated? You must know everything"
    if re.search(r"you must (know|be|have|think|believe)", t):
        return True

    # Pattern 4: Implied impossibility / absurdity
    impossibility_phrases = [
        "like that'll ever happen",
        "like that will ever happen",
        "like that ever works",
        "as if that",
        "oh i'm sure that",
        "i'm sure that'll",
        "right, because",
        "sure, because",
        "oh because",
    ]
    if any(p in t for p in impossibility_phrases):
        return True

    # Pattern 5: "Nothing like X" sarcastic opener
    if re.search(r"^nothing like", t):
        return True

    # Pattern 6: Exaggerated self-deprecation or praise
    exaggeration_markers = [
        "best day ever",
        "worst day ever",
        "most helpful",
        "most useful thing",
        "really helped",
        "oh this is fine",
        "this is fine",
        "totally fine",
    ]
    if any(p in t for p in exaggeration_markers):
        return True

    return False


def detect_contrast_sarcasm(text):
    """
    Detects sentiment flip after contrast words (but / however / although).
    If the first clause is positive and the second contains negative words,
    the sentence likely expresses negative sentiment with a sarcastic edge.
    Returns: "NEGATIVE" | "POSITIVE" | None
    """
    t = text.lower()
    contrast_words = ["but", "however", "although", "though", "yet", "except"]

    negative_words = [
        "hot", "bad", "worst", "problem", "slow", "delay", "terrible",
        "poor", "awful", "horrible", "broken", "failed", "crash",
        "error", "impossible", "useless", "annoying"
    ]
    positive_words = [
        "good", "great", "nice", "fast", "amazing", "excellent",
        "perfect", "love", "best", "wonderful"
    ]

    for word in contrast_words:
        if word in t:
            parts = t.split(word, 1)
            if len(parts) == 2:
                first, second = parts
                first_positive = any(p in first for p in positive_words)
                second_negative = any(n in second for n in negative_words)
                if first_positive and second_negative:
                    return "NEGATIVE"
                first_negative = any(n in first for n in negative_words)
                second_positive = any(p in second for p in positive_words)
                if first_negative and second_positive:
                    return "POSITIVE"
    return None


# Final Combined Prediction Function
def predict_final(text):
    """
    Combines:
      1. TF-IDF + Logistic Regression (ML)
      2. Keyword rule-based sarcasm
      3. Rhetorical / conversational sarcasm patterns  ← NEW
      4. Contrast/adversative sentiment flip           ← IMPROVED
      5. Transformer sentiment (DistilBERT)
    """
    # --- Preprocessing ---
    clean = clean_text(text)
    vec = vectorizer.transform([clean])

    # --- Layer 1: ML prediction ---
    sarcasm_ml = model.predict(vec)[0]

    # --- Layer 2: Keyword rules ---
    sarcasm_rule = is_rule_based_sarcasm(text)

    # --- Layer 3: Rhetorical sarcasm (NEW) ---
    sarcasm_rhetorical = detect_rhetorical_sarcasm(text)

    # --- Final sarcasm verdict ---
    final_sarcasm = bool(sarcasm_ml == 1 or sarcasm_rule or sarcasm_rhetorical)

    # --- Layer 4: Transformer sentiment ---
    sentiment_result = sentiment_model(text)[0]
    sentiment = sentiment_result['label']        # 'POSITIVE' or 'NEGATIVE'
    confidence = sentiment_result['score']

    # --- Layer 5: Contrast sentiment override ---
    contrast_result = detect_contrast_sarcasm(text)
    if contrast_result:
        sentiment = contrast_result

    # --- Sarcasm overrides sentiment to NEGATIVE ---
    if final_sarcasm:
        sentiment = "NEGATIVE"

    return {
        "input_text": text,
        "sarcasm": "Yes" if final_sarcasm else "No",
        "final_sentiment": sentiment,
        "confidence": round(confidence, 3),
        "signals": {
            "ml_model": bool(sarcasm_ml == 1),
            "keyword_rule": sarcasm_rule,
            "rhetorical_pattern": sarcasm_rhetorical,   # NEW
            "contrast_flip": contrast_result is not None,
        }
    }


# Test Suite

test_sentences = [
    # Previously failing cases (rhetorical sarcasm)
    "I'd agree with you, but then we'd both be wrong",
    "Oh, you just graduated? You must know everything.",
    "Yeah right, like that'll ever happen.",
    "Oh sure, I totally meant to do that.",

    # Clear sarcasm
    "Great, my phone died again",
    "Wow amazing service, 2 hour delay",
    "Perfect timing, the app crashed again",
    "Fantastic, another bug in the system",
    "Nice, my internet stopped working",

    # Subtle sarcasm
    "Just what I needed, more problems",
    "Love when my laptop freezes",
    "Amazing, it broke on the first day",
    "Nothing like a Monday morning crash to start the week",

    # Genuine positive
    "I absolutely love this phone",
    "This product is amazing",
    "The movie was fantastic",

    # Genuine negative
    "This is the worst service ever",
    "I hate this app",
    "Very bad experience",

    # Mixed / contrast
    "The design is great but performance is terrible",
    "Good features but very slow",

    # Ambiguous / neutral
    "This is interesting",
    "Well that was something",
    "The phone was released yesterday",
    "It is a device used for communication",
]

print("=" * 70)
for s in test_sentences:
    result = predict_final(s)
    print(f"Text     : {result['input_text']}")
    print(f"Sarcasm  : {result['sarcasm']}  |  Sentiment: {result['final_sentiment']}  |  Conf: {result['confidence']}")
    print(f"Signals  : {result['signals']}")
    print("-" * 70)

(26709, 3)
is_sarcastic
0    14985
1    11724
Name: count, dtype: int64
ML Model Accuracy: 0.8384500187195807
              precision    recall  f1-score   support

           0       0.85      0.87      0.86      2996
           1       0.83      0.80      0.81      2346

    accuracy                           0.84      5342
   macro avg       0.84      0.83      0.84      5342
weighted avg       0.84      0.84      0.84      5342



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

Text     : I'd agree with you, but then we'd both be wrong
Sarcasm  : No  |  Sentiment: NEGATIVE  |  Conf: 0.986
Signals  : {'ml_model': False, 'keyword_rule': False, 'rhetorical_pattern': False, 'contrast_flip': False}
----------------------------------------------------------------------
Text     : Oh, you just graduated? You must know everything.
Sarcasm  : Yes  |  Sentiment: NEGATIVE  |  Conf: 0.801
Signals  : {'ml_model': False, 'keyword_rule': False, 'rhetorical_pattern': True, 'contrast_flip': False}
----------------------------------------------------------------------
Text     : Yeah right, like that'll ever happen.
Sarcasm  : Yes  |  Sentiment: NEGATIVE  |  Conf: 1.0
Signals  : {'ml_model': True, 'keyword_rule': True, 'rhetorical_pattern': True, 'contrast_flip': False}
----------------------------------------------------------------------
Text     : Oh sure, I totally meant to do that.
Sarcasm  : Yes  |  Sentiment: NEGATIVE  |  Conf: 0.999
Signals  : {'ml_model': True, 'keywo